In [1]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: CĂN CHỈNH ẢNH BẰNG SIFT, MATCHING VÀ HOMOGRAPHY
# Chương/Mục liên quan: Chương 6 - Căn chỉnh ảnh
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa bài toán căn chỉnh hai ảnh có vùng quan sát chồng lấn.
# - Sử dụng SIFT để phát hiện và mô tả điểm đặc trưng.
# - Sử dụng Lowe ratio test để chọn các cặp điểm tương ứng tốt.
# - Ước lượng homography bằng RANSAC để căn chỉnh hai ảnh.

# Sau khi chạy code, người học cần:
# 1. Quan sát được các điểm đặc trưng SIFT trên từng ảnh.
# 2. Quan sát được các cặp điểm khớp giữa hai ảnh.
# 3. Hiểu được vai trò của RANSAC trong việc loại bỏ outlier.
# 4. Quan sát được kết quả căn chỉnh ảnh dựa trên homography.

# Input:
# - Dữ liệu đầu vào: hai ảnh màu từ GitHub
# - Kiểu dữ liệu: ảnh màu

# Output:
# - Ảnh với các điểm đặc trưng SIFT
# - Ảnh thể hiện các cặp điểm khớp tốt
# - Ma trận homography H_21
# - Ảnh căn chỉnh cuối cùng có đánh dấu inlier và outlier

# Lưu ý
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# Người học không chỉ chạy code để xem kết quả, mà cần hiểu:
# - Code đang minh họa nội dung lý thuyết nào;
# - Các bước xử lý tương ứng với công thức hoặc thuật toán nào;
# - Kết quả đầu ra có hợp lý hay không;
# - Khi thay đổi tham số, kết quả thay đổi như thế nào.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen


# ============================================================
# 2. TẢI DỮ LIỆU ĐẦU VÀO
# ============================================================

def read_image_from_url(image_url):
    resp = urlopen(image_url)
    image_data = np.asarray(bytearray(resp.read()), dtype=np.uint8)
    I_bgr = cv2.imdecode(image_data, cv2.IMREAD_COLOR)

    if I_bgr is None:
        raise ValueError(f"Không đọc được ảnh từ URL: {image_url}")

    return I_bgr


# ============================================================
# 3. HIỂN THỊ ẢNH ĐẦU VÀO
# ============================================================

def show_bgr(I_bgr, title="", figsize=(10, 8)):
    I_rgb = cv2.cvtColor(I_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=figsize)
    plt.imshow(I_rgb)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# ============================================================
# 4. XỬ LÝ CHÍNH
# ============================================================

def enhance_contrast(I_bgr):
    I_lab = cv2.cvtColor(I_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(I_lab)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    L = clahe.apply(L)

    I_lab = cv2.merge((L, A, B))
    return cv2.cvtColor(I_lab, cv2.COLOR_LAB2BGR)


def detect_and_describe_sift(I_gray):
    sift = cv2.SIFT_create()
    keypoints, descriptors = sift.detectAndCompute(I_gray, None)
    return keypoints, descriptors


def select_strong_keypoints(keypoints, max_points=250):
    keypoints = sorted(keypoints, key=lambda kp: kp.response, reverse=True)
    return keypoints[:max_points]


def draw_sift_keypoints(I_bgr, keypoints, max_points=250):
    I_out = I_bgr.copy()
    keypoints_show = select_strong_keypoints(keypoints, max_points=max_points)

    for kp in keypoints_show:
        x, y = int(kp.pt[0]), int(kp.pt[1])
        r = max(4, int(kp.size * 0.35))

        cv2.circle(I_out, (x, y), r, (0, 255, 255), 2)
        cv2.circle(I_out, (x, y), 3, (0, 0, 255), -1)

    return I_out


def match_features_ratio_test(descriptors1, descriptors2, ratio_thresh=0.75):
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    knn_matches = bf.knnMatch(descriptors1, descriptors2, k=2)

    good_matches = []

    for pair in knn_matches:
        if len(pair) < 2:
            continue

        m, n = pair

        # Dòng này tương ứng với Lowe ratio test:
        # chỉ giữ match m nếu khoảng cách của m nhỏ hơn ratio_thresh lần khoảng cách của n.
        if m.distance < ratio_thresh * n.distance:
            good_matches.append(m)

    good_matches = sorted(good_matches, key=lambda m: m.distance)
    return good_matches


def draw_top_matches(I1_bgr, I2_bgr, keypoints1, keypoints2, matches, top_k=20):
    matches = matches[:top_k]

    H1, W1 = I1_bgr.shape[:2]
    H2, W2 = I2_bgr.shape[:2]

    canvas = np.zeros((max(H1, H2), W1 + W2, 3), dtype=np.uint8)
    canvas[:H1, :W1] = I1_bgr
    canvas[:H2, W1:] = I2_bgr

    for m in matches:
        x1, y1 = map(int, keypoints1[m.queryIdx].pt)
        x2, y2 = map(int, keypoints2[m.trainIdx].pt)
        x2_shifted = x2 + W1

        cv2.line(canvas, (x1, y1), (x2_shifted, y2), (0, 255, 0), 3)
        cv2.circle(canvas, (x1, y1), 6, (0, 0, 255), -1)
        cv2.circle(canvas, (x2_shifted, y2), 6, (255, 0, 0), -1)

    return canvas


def estimate_homography_ransac(keypoints1, keypoints2, matches, ransac_thresh=3.0):
    if len(matches) < 4:
        raise ValueError("Không đủ số cặp điểm để ước lượng homography.")

    points1 = np.float32([keypoints1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    points2 = np.float32([keypoints2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Dòng này tương ứng với bước ước lượng homography bằng RANSAC.
    H_21, inlier_mask = cv2.findHomography(points2, points1, cv2.RANSAC, ransac_thresh)

    if H_21 is None:
        raise ValueError("Không ước lượng được homography.")

    return H_21, inlier_mask.ravel().astype(bool)


def compute_output_canvas(I1_bgr, I2_bgr, H_21):
    H1, W1 = I1_bgr.shape[:2]
    H2, W2 = I2_bgr.shape[:2]

    corners_I1 = np.float32([
        [0, 0],
        [W1 - 1, 0],
        [W1 - 1, H1 - 1],
        [0, H1 - 1]
    ]).reshape(-1, 1, 2)

    corners_I2 = np.float32([
        [0, 0],
        [W2 - 1, 0],
        [W2 - 1, H2 - 1],
        [0, H2 - 1]
    ]).reshape(-1, 1, 2)

    corners_I2_warped = cv2.perspectiveTransform(corners_I2, H_21)

    all_corners = np.vstack((corners_I1, corners_I2_warped))
    x_all = all_corners[:, 0, 0]
    y_all = all_corners[:, 0, 1]

    x_min = int(np.floor(np.min(x_all)))
    y_min = int(np.floor(np.min(y_all)))
    x_max = int(np.ceil(np.max(x_all)))
    y_max = int(np.ceil(np.max(y_all)))

    tx = -x_min if x_min < 0 else 0
    ty = -y_min if y_min < 0 else 0

    T = np.array([
        [1, 0, tx],
        [0, 1, ty],
        [0, 0, 1]
    ], dtype=np.float64)

    output_width = x_max - x_min + 1
    output_height = y_max - y_min + 1

    return T, output_width, output_height


def draw_alignment_full_canvas(
    I1_bgr, I2_bgr,
    keypoints1, keypoints2,
    matches, inlier_mask,
    H_21,
    point_radius=7
):
    H1, W1 = I1_bgr.shape[:2]
    H2, W2 = I2_bgr.shape[:2]

    T, output_width, output_height = compute_output_canvas(I1_bgr, I2_bgr, H_21)

    H_canvas = T @ H_21

    # Dòng này thực hiện phép biến đổi phối cảnh dựa trên homography.
    I2_warped = cv2.warpPerspective(I2_bgr, H_canvas, (output_width, output_height))

    canvas = np.full((output_height, output_width, 3), 235, dtype=np.uint8)

    tx = int(round(T[0, 2]))
    ty = int(round(T[1, 2]))

    canvas[ty:ty + H1, tx:tx + W1] = I1_bgr

    mask_I2 = cv2.cvtColor(I2_warped, cv2.COLOR_BGR2GRAY) > 0
    alpha = 0.55

    canvas[mask_I2] = (
        (1.0 - alpha) * canvas[mask_I2].astype(np.float32)
        + alpha * I2_warped[mask_I2].astype(np.float32)
    ).astype(np.uint8)

    corners_I1 = np.float32([
        [0, 0],
        [W1 - 1, 0],
        [W1 - 1, H1 - 1],
        [0, H1 - 1]
    ]).reshape(-1, 1, 2)

    corners_I1_canvas = cv2.perspectiveTransform(corners_I1, T)
    corners_I1_canvas = np.round(corners_I1_canvas).astype(np.int32)
    cv2.polylines(canvas, [corners_I1_canvas], True, (0, 0, 0), 3)

    corners_I2 = np.float32([
        [0, 0],
        [W2 - 1, 0],
        [W2 - 1, H2 - 1],
        [0, H2 - 1]
    ]).reshape(-1, 1, 2)

    corners_I2_canvas = cv2.perspectiveTransform(corners_I2, H_canvas)
    corners_I2_canvas = np.round(corners_I2_canvas).astype(np.int32)
    cv2.polylines(canvas, [corners_I2_canvas], True, (255, 0, 255), 3)

    points2 = np.float32(
        [keypoints2[m.trainIdx].pt for m in matches]
    ).reshape(-1, 1, 2)

    points2_canvas = cv2.perspectiveTransform(points2, H_canvas).reshape(-1, 2)

    for i, p in enumerate(points2_canvas):
        if inlier_mask[i] and np.random.rand() < (2 / 3):
            continue

        x, y = int(round(p[0])), int(round(p[1]))

        if 0 <= x < output_width and 0 <= y < output_height:
            if inlier_mask[i]:
                color = (0, 255, 0)
            else:
                color = (0, 0, 255)

            cv2.circle(canvas, (x, y), point_radius + 2, (255, 255, 255), -1)
            cv2.circle(canvas, (x, y), point_radius, color, -1)
            cv2.circle(canvas, (x, y), point_radius, (0, 0, 0), 1)

    return canvas

def main():
    base_url = "https://github.com/lthavnu/cv-book/raw/main/images/"
    url_I1 = base_url + "Oulu1.png"
    url_I2 = base_url + "Oulu2.png"

    I1_bgr = read_image_from_url(url_I1)
    I2_bgr = read_image_from_url(url_I2)

    I1_bgr = enhance_contrast(I1_bgr)
    I2_bgr = enhance_contrast(I2_bgr)

    I1_gray = cv2.cvtColor(I1_bgr, cv2.COLOR_BGR2GRAY)
    I2_gray = cv2.cvtColor(I2_bgr, cv2.COLOR_BGR2GRAY)

    keypoints1, descriptors1 = detect_and_describe_sift(I1_gray)
    keypoints2, descriptors2 = detect_and_describe_sift(I2_gray)

    I1_kp = draw_sift_keypoints(I1_bgr, keypoints1, max_points=500)
    I2_kp = draw_sift_keypoints(I2_bgr, keypoints2, max_points=500)
# ============================================================
# 5. HIỂN THỊ KẾT QUẢ
# ============================================================

    show_bgr(I1_kp, "", figsize=(8, 6))
    show_bgr(I2_kp, "", figsize=(8, 6))

    matches = match_features_ratio_test(descriptors1, descriptors2, ratio_thresh=0.75)

    match_bgr = draw_top_matches(
        I1_bgr, I2_bgr,
        keypoints1, keypoints2,
        matches,
        top_k=20
    )

    show_bgr(match_bgr, "", figsize=(14, 7))

    H_21, inlier_mask = estimate_homography_ransac(
        keypoints1, keypoints2, matches, ransac_thresh=3.0
    )

    # ============================================================
    # 6. KIỂM TRA KẾT QUẢ
    # ============================================================

    print("Số keypoints I1:", len(keypoints1))
    print("Số keypoints I2:", len(keypoints2))
    print("Số matches tốt   :", len(matches))
    print("Số inliers       :", int(np.sum(inlier_mask)))
    print("Số outliers      :", int(len(inlier_mask) - np.sum(inlier_mask)))
    print("\nHomography H_21 =")
    print(H_21)

    alignment_bgr = draw_alignment_full_canvas(
        I1_bgr, I2_bgr,
        keypoints1, keypoints2,
        matches, inlier_mask,
        H_21,
        point_radius=7
    )

    show_bgr(alignment_bgr, "", figsize=(10, 8))

    # ============================================================
    # 7. GỢI Ý THỬ NGHIỆM CHO NGƯỜI HỌC
    # ============================================================

    # Có thể thay đổi max_points trong draw_sift_keypoints để quan sát nhiều hoặc ít điểm đặc trưng hơn.
    # Có thể thay đổi ratio_thresh trong match_features_ratio_test để kiểm soát mức nghiêm ngặt của Lowe ratio test.
    # Có thể thay đổi ransac_thresh trong estimate_homography_ransac để quan sát ảnh hưởng đến số inlier và outlier.


if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.